# 01 - Deploy Infrastructure

This notebook deploys `infra/main.bicep` and writes the core deployment outputs into `env/.env` so later notebooks can run without manual value copy/paste.

**Estimated time:** 10-20 minutes

---

## What gets deployed

- Azure AI Services account (`kind: AIServices`)
- Foundry project (`Microsoft.CognitiveServices/accounts/projects`)
- Azure Function App + App Insights for later function publishing

---

## Step 1 - Set deployment variables

In [7]:
RESOURCE_GROUP = "rg-speech01-foundry"
LOCATION = "eastus2"
DEMO_ID = "speech01"

print(f"Resource group : {RESOURCE_GROUP}")
print(f"Location       : {LOCATION}")
print(f"Demo ID        : {DEMO_ID}")


Resource group : rg-speech01-foundry
Location       : eastus2
Demo ID        : speech01


## Step 2 - Create the resource group

In [8]:
!az group create \
    --name {RESOURCE_GROUP} \
    --location {LOCATION} \
    --output table

Location    Name
----------  -------------------
eastus2     rg-speech01-foundry


## Step 3 - Deploy Bicep template

In [9]:
import shutil
import subprocess
from pathlib import Path


def resolve_az_cli():
    az_in_path = shutil.which("az")
    if az_in_path:
        return az_in_path
    for candidate in [
        Path(r"C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
        Path(r"C:\\Program Files (x86)\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
    ]:
        if candidate.exists():
            return str(candidate)
    return None


def run_deployment(az_cmd: str, include_deployer_principal: bool, deployer_object_id: str):
    deploy_command = [
        az_cmd,
        "deployment",
        "group",
        "create",
        "--resource-group",
        RESOURCE_GROUP,
        "--template-file",
        "../infra/main.bicep",
        "--parameters",
        f"location={LOCATION}",
        f"demoId={DEMO_ID}",
    ]
    if include_deployer_principal and deployer_object_id:
        deploy_command.append(f"deployerPrincipalId={deployer_object_id}")
    deploy_command.extend(["--output", "table"])

    return subprocess.run(deploy_command, capture_output=True, text=True)


az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Install it or add it to PATH.")

deployer_object_id_result = subprocess.run(
    [az_cmd, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True,
    text=True,
 )
deployer_object_id = deployer_object_id_result.stdout.strip()
if deployer_object_id:
    print(f"Using deployer principal ID for storage upload RBAC: {deployer_object_id}")
else:
    print(
        "Could not detect signed-in user object ID automatically; continuing without "
        "deployerPrincipalId."
    )

# Deploy the core infrastructure for the prototype.
redeploy_result = run_deployment(
    az_cmd=az_cmd,
    include_deployer_principal=bool(deployer_object_id),
    deployer_object_id=deployer_object_id,
 )
print(redeploy_result.stdout)
if redeploy_result.returncode != 0:
    stderr_text = redeploy_result.stderr.strip()
    if "RoleAssignmentExists" in stderr_text and deployer_object_id:
        print(
            "Detected existing storage RBAC assignment with a different role-assignment ID. "
            "Retrying deployment without deployerPrincipalId."
        )
        redeploy_result = run_deployment(
            az_cmd=az_cmd,
            include_deployer_principal=False,
            deployer_object_id=deployer_object_id,
        )
        print(redeploy_result.stdout)

    if redeploy_result.returncode != 0:
        raise RuntimeError(
            f"Bicep deployment failed.\n"
            f"stderr: {redeploy_result.stderr.strip()}\n"
            f"stdout: {redeploy_result.stdout.strip()}"
        )


Using deployer principal ID for storage upload RBAC: d1268ed1-a5e6-4c66-815f-8d3b21d300e9
Name    State      Timestamp                         Mode         ResourceGroup
------  ---------  --------------------------------  -----------  -------------------
main    Succeeded  2026-06-05T15:58:47.894354+00:00  Incremental  rg-speech01-foundry



## Step 4 - Retrieve deployment outputs

Query the latest deployment in this resource group and capture the values needed for the later notebooks.

In [10]:
import json
import shutil
import subprocess
from pathlib import Path


def resolve_az_cli():
    az_in_path = shutil.which("az")
    if az_in_path:
        return az_in_path
    windows_fallbacks = [
        Path(r"C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
        Path(r"C:\\Program Files (x86)\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
    ]
    for candidate in windows_fallbacks:
        if candidate.exists():
            return str(candidate)
    return None


az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found from this kernel. Install it or add it to PATH.")

result = subprocess.run(
    [
        az_cmd,
        "deployment",
        "group",
        "list",
        "--resource-group",
        RESOURCE_GROUP,
        "--query",
        "[0].properties.outputs",
        "--output",
        "json",
    ],
    capture_output=True,
    text=True,
    check=True,
)

outputs = json.loads(result.stdout)

ai_account_name = outputs["aiServicesAccountName"]["value"]
ai_endpoint = outputs["aiServicesEndpoint"]["value"]
auth_mode = "aad"
function_app_name = outputs.get("functionAppName", {}).get("value", "")
function_url = outputs.get("functionUrl", {}).get("value", "")
app_insights_name = outputs.get("appInsightsName", {}).get("value", "")

print(f"AI account             : {ai_account_name}")
print(f"AI endpoint            : {ai_endpoint}")
print(f"Auth mode              : {auth_mode}")
print(f"Function app           : {function_app_name or '(not yet deployed)'}")
print(f"Function URL           : {function_url or '(not yet available)'}")
print(f"App Insights           : {app_insights_name or '(not yet deployed)'}")

AI account             : aispeech016m4wo
AI endpoint            : https://aispeech016m4wo.cognitiveservices.azure.com/
Auth mode              : aad
Function app           : func-speech01-6m4wo
Function URL           : https://func-speech01-6m4wo.azurewebsites.net/api/speech-roundtrip
App Insights           : appi-speech01-6m4wo


## Step 5 - Write outputs to `env/.env`

This persists values from Step 4 for the configure and test notebooks.

> **Security note:** `env/.env` is git-ignored. Never commit it.

In [11]:
import pathlib

env_lines = [
    "# Auto-generated by 01_deploy_infra.ipynb - do not commit",
    f"AZURE_RESOURCE_GROUP={RESOURCE_GROUP}",
    f"AZURE_AI_ENDPOINT={ai_endpoint}",
    f"AZURE_AUTH_MODE={auth_mode}",
    "AZURE_SPEECH_VOICE=en-US-AvaMultilingualNeural",
    f"AZURE_FUNCTION_APP_NAME={function_app_name}",
    f"AZURE_FUNCTION_URL={function_url}",
    f"AZURE_APP_INSIGHTS_NAME={app_insights_name}",
    "AZURE_FUNCTION_KEY=",
]

env_content = "\n".join(env_lines) + "\n"

env_path = pathlib.Path("../env/.env")
env_path.write_text(env_content, encoding="utf-8")
print(f"✅ Written to {env_path.resolve()}")

✅ Written to C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demos\speech01-foundry-speech\env\.env


## Step 5 - Verify `env/.env`

Confirm the generated environment file exists and contains the bootstrap values needed by the later notebooks.


In [12]:
import pathlib

env_path = pathlib.Path("../env/.env")
example_path = pathlib.Path("../env/.env.example")

if env_path.exists():
    print("env/.env found")
    print(f"Path: {env_path.resolve()}")
    env_text = env_path.read_text(encoding="utf-8")
    if "AZURE_FUNCTION_KEY=" in env_text:
        print("Notebook 2 will populate AZURE_FUNCTION_KEY after the Function App code is published.")
else:
    print("ERROR: env/.env not found")
    print("Step 5 should generate this file.")
    print(f"Expected variable contract from {example_path}:")
    print(example_path.read_text(encoding="utf-8"))
    raise FileNotFoundError(env_path)

env/.env found
Path: C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demos\speech01-foundry-speech\env\.env
Notebook 2 will populate AZURE_FUNCTION_KEY after the Function App code is published.


---

## ✅ Infrastructure deployed

Move to **`02_configure.ipynb`** to validate Speech readiness and inspect voices.

---

## Tear-down

In [ ]:
# Uncomment and run to delete all resources when finished.
# !az group delete --name {RESOURCE_GROUP} --yes --no-wait
# print("Resource group deletion triggered.")